# Flickr30k T3→S3 Direct Logit-KD — seed 42

Teacher SigLIP đã thích nghi decoder Flickr30k; Teacher và Student dùng chung
vocabulary 7.456 token. Loss khóa:
`L = 0,5 CE_human + 0,5 T² KL(Teacher || Student)`, `T = 4`.

## Kaggle input bắt buộc — đúng 8 input

1. Output `F30K_Data`.
2. Output `F30K_Base`.
3. Output `F30K_S3_42`.
4. Output `F30K_T3_Adapt42_Full`.
5. Output `phase3_teacher_c_seed42`.
6. Dataset `Flickr Image dataset`.
7. Dataset/Kaggle Model SigLIP offline có `MODEL_SOURCE_MANIFEST.json`.
8. Package/dataset `pycocoevalcap-1.2` offline.

Khi resume, gắn thêm đúng một output gần nhất của notebook `F30K_T3S3_LogitKD_42`.

Runtime: GPU T4, Internet Off. Dự kiến 4–8 giờ; physical/accum/effective
batch `8/8/64`; tối đa 20 epoch; Early Stopping từ epoch 5, patience 4.
Test luôn `SEALED / NOT USED`.

## Cell 1 — Authorization, integrity và Test-seal

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"


import csv, hashlib, importlib.util, json, math, os, random, shutil, subprocess, sys, time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

MODEL_ID = "S3"
MODEL_NAME = "TinyViT-5M"
SEED = 42
EXPERIMENT_ID = "F30K_T3S3_LogitKD_42"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def unique_named(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (f"Cần đúng một {name}", matches)
    return matches[0]

BASE_READY_PATH = unique_named("F30K_BASE_READY.json")
DATA_CONTRACT_PATH = unique_named("F30K_DATA_CONTRACT.json")
PROTOCOL_PATH = unique_named("F30K_BASELINE_PROTOCOL.json")
TRAIN_PATH = unique_named("f30k_train.parquet")
VAL_PATH = unique_named("f30k_validation.parquet")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")

base_ready = json.loads(BASE_READY_PATH.read_text(encoding="utf-8"))
data_contract = json.loads(DATA_CONTRACT_PATH.read_text(encoding="utf-8"))
protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
vocab = json.loads(VOCAB_PATH.read_text(encoding="utf-8"))

assert base_ready["status"] == "authorized"
assert base_ready["test_set_status"] == "sealed"
assert base_ready["test_set_used"] is False
assert (
    SEED == int(protocol["development_seed"])
    or SEED in [int(x) for x in protocol["confirmation_seeds"]]
)
assert protocol["vocabulary_size"] == 7456 == len(vocab["tokens"])
assert protocol["maximum_epochs"] == 20
assert protocol["selection_metric"] == "Validation CIDEr-D"
assert data_contract["test_set_status"] == "sealed"
assert data_contract["test_caption_exported"] is False

for file_name, expected in {
    "f30k_train.parquet": data_contract["artifacts"]["train"]["sha256"],
    "f30k_validation.parquet": data_contract["artifacts"]["validation"]["sha256"],
    "f30k_vocabulary_minfreq5.json": data_contract["artifacts"]["vocabulary"]["sha256"],
}.items():
    actual_path = {"f30k_train.parquet": TRAIN_PATH,
                   "f30k_validation.parquet": VAL_PATH,
                   "f30k_vocabulary_minfreq5.json": VOCAB_PATH}[file_name]
    assert sha256(actual_path) == expected, file_name

if importlib.util.find_spec("pycocoevalcap") is None:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    if wheels:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])])
    else:
        packages = [p for p in INPUT_ROOT.rglob("pycocoevalcap") if p.is_dir()]
        assert packages, "Hãy gắn input pycocoevalcap-1.2 offline."
        sys.path.insert(0, str(packages[0].parent))

assert importlib.util.find_spec("pycocoevalcap") is not None
assert shutil.which("java"), "METEOR chính thức yêu cầu Java runtime."
assert torch.cuda.is_available(), "Notebook này yêu cầu GPU T4."
DEVICE = torch.device("cuda")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("Preflight             : PASSED")
print("Experiment            :", EXPERIMENT_ID)
print("Model                 :", MODEL_NAME)
print("Train/Validation      : VERIFIED")
print("Vocabulary            :", len(vocab["tokens"]))
print("Selection             : Validation CIDEr-D")
print("Test                   : SEALED / NOT READ")

Preflight             : PASSED
Experiment            : F30K_T3S3_LogitKD_42
Model                 : TinyViT-5M
Train/Validation      : VERIFIED
Vocabulary            : 7456
Selection             : Validation CIDEr-D
Test                   : SEALED / NOT READ


## Cell 2 — Student architecture và Flickr30k loaders

In [2]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
assert train_df["image_id"].nunique() == 29000 and len(train_df) == 145000
assert val_df["image_id"].nunique() == 1014 and len(val_df) == 5070
assert set(train_df.image_id).isdisjoint(set(val_df.image_id))

needed_names = {f"{int(x)}.jpg" for x in pd.concat([train_df.image_id, val_df.image_id])}
image_candidates = [p for p in INPUT_ROOT.rglob("*.jpg") if p.name in needed_names]
IMAGE_BY_NAME = {}
duplicate_alias_count = 0
for path in image_candidates:
    old = IMAGE_BY_NAME.get(path.name)
    if old is None:
        IMAGE_BY_NAME[path.name] = path
        continue
    if old.resolve() == path.resolve():
        continue
    # Một số bản đóng gói Kaggle lưu cùng ảnh ở cả
    # flickr30k_images/<file> và flickr30k_images/flickr30k_images/<file>.
    # Chấp nhận alias lồng nhau khi kích thước byte giống hệt và luôn chọn
    # đường dẫn nông/ngắn nhất. Nếu kích thước khác nhau, dừng để tránh ánh xạ
    # nhầm hai ảnh chỉ tình cờ có cùng tên.
    assert old.stat().st_size == path.stat().st_size, (
        "Hai ảnh cùng tên nhưng khác kích thước", path.name, old, path
    )
    duplicate_alias_count += 1
    IMAGE_BY_NAME[path.name] = min(
        [old, path],
        key=lambda p: (len(p.parts), len(str(p)), str(p)),
    )
missing = sorted(needed_names - set(IMAGE_BY_NAME))
assert not missing, ("Thiếu ảnh Flickr30k", len(missing), missing[:10])

train_groups = []
for image_id, group in train_df.groupby("image_id", sort=True):
    train_groups.append((int(image_id), [list(map(int, x)) for x in group.token_ids]))
val_groups = []
VAL_REFERENCES = {}
for image_id, group in val_df.groupby("image_id", sort=True):
    iid = int(image_id)
    val_groups.append((iid, [list(map(int, x)) for x in group.token_ids]))
    VAL_REFERENCES[iid] = [str(x) for x in group.caption.tolist()]

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CaptionDataset(Dataset):
    def __init__(self, rows, transform, training):
        self.rows, self.transform, self.training, self.epoch = rows, transform, training, 0
    def set_epoch(self, epoch):
        self.epoch = int(epoch)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        image_id, captions = self.rows[index]
        caption = captions[(self.epoch + image_id % len(captions)) % len(captions)] if self.training else captions[0]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as im:
            image = self.transform(im.convert("RGB"))
        return image_id, image, torch.tensor(caption, dtype=torch.long)

def collate(batch):
    image_ids, images, captions = zip(*batch)
    max_len = max(len(x) for x in captions)
    tokens = torch.zeros(len(captions), max_len, dtype=torch.long)
    mask = torch.zeros(len(captions), max_len, dtype=torch.bool)
    for i, x in enumerate(captions):
        tokens[i, :len(x)] = x
        mask[i, :len(x)] = True
    return torch.tensor(image_ids), torch.stack(images), tokens, mask

train_set = CaptionDataset(train_groups, train_transform, True)
val_set = CaptionDataset(val_groups, val_transform, False)

def make_train_loader(epoch):
    train_set.set_epoch(epoch)
    g = torch.Generator().manual_seed(SEED + epoch)
    return DataLoader(train_set, batch_size=32, shuffle=True, generator=g, num_workers=2,
                      pin_memory=True, persistent_workers=False, collate_fn=collate)

val_loader = DataLoader(val_set, batch_size=64, shuffle=False, num_workers=2,
                        pin_memory=True, persistent_workers=True, collate_fn=collate)

def select_tensor(x):
    if torch.is_tensor(x):
        return x
    if isinstance(x, dict):
        return [v for v in x.values() if torch.is_tensor(v)][-1]
    if isinstance(x, (tuple, list)):
        return [v for v in x if torch.is_tensor(v)][-1]
    raise TypeError(type(x))

class VisualEncoder(nn.Module):
    def __init__(self, backbone, source_dim):
        super().__init__()
        self.backbone = backbone
        self.source_dim = source_dim
        self.projection = nn.Linear(source_dim, 256)
        self.visual_position_embedding = nn.Parameter(torch.empty(1, 49, 256))
        self.output_norm = nn.LayerNorm(256)
        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.zeros_(self.projection.bias)
        nn.init.normal_(self.visual_position_embedding, std=0.02)
    def forward(self, images):
        if MODEL_ID == "S1":
            feat = self.backbone(images)
        else:
            feat = select_tensor(self.backbone.forward_features(images))
        assert feat.ndim == 4 and feat.shape[1] == self.source_dim and tuple(feat.shape[-2:]) == (7, 7), feat.shape
        tokens = feat.flatten(2).transpose(1, 2).contiguous()
        return self.output_norm(self.projection(tokens) + self.visual_position_embedding)

class CaptionDecoder(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.d_model, self.max_steps = 256, 31
        self.token_embedding = nn.Embedding(vocabulary_size, 256, padding_idx=0)
        self.position_embedding = nn.Embedding(31, 256)
        self.embedding_dropout = nn.Dropout(0.1)
        layer = nn.TransformerDecoderLayer(256, 8, 1024, 0.1, "gelu",
                                           batch_first=True, norm_first=True)
        self.transformer_decoder = nn.TransformerDecoder(layer, 3)
        self.final_norm = nn.LayerNorm(256)
        self.output_bias = nn.Parameter(torch.zeros(vocabulary_size))
        nn.init.normal_(self.token_embedding.weight, std=256 ** -0.5)
        nn.init.normal_(self.position_embedding.weight, std=0.02)
        with torch.no_grad():
            self.token_embedding.weight[0].zero_()
    def forward(self, token_ids, memory, attention_mask=None):
        n, t = token_ids.shape
        pos = torch.arange(t, device=token_ids.device).unsqueeze(0).expand(n, -1)
        x = self.embedding_dropout(self.token_embedding(token_ids) * 16.0 + self.position_embedding(pos))
        causal = torch.triu(torch.ones(t, t, dtype=torch.bool, device=token_ids.device), diagonal=1)
        key_pad = None if attention_mask is None else ~attention_mask.bool()
        x = self.transformer_decoder(x, memory, tgt_mask=causal, tgt_key_padding_mask=key_pad)
        return F.linear(self.final_norm(x), self.token_embedding.weight, self.output_bias)

if MODEL_ID == "S1":
    from torchvision.models import mobilenet_v3_large
    base = mobilenet_v3_large(weights=None).features
elif MODEL_ID == "S2":
    import timm
    base = timm.create_model("mobilevit_s.cvnets_in1k", pretrained=False)
    base.reset_classifier(num_classes=0, global_pool="")
    for head_name in ["head", "head_dist", "classifier", "fc"]:
        head = getattr(base, head_name, None)
        if isinstance(head, nn.Module) and not isinstance(head, nn.Identity):
            setattr(base, head_name, nn.Identity())
elif MODEL_ID == "S3":
    import timm
    base = timm.create_model("tiny_vit_5m_224.dist_in22k_ft_in1k", pretrained=False)
    base.reset_classifier(num_classes=0, global_pool="")
    for head_name in ["head", "head_dist", "classifier", "fc"]:
        head = getattr(base, head_name, None)
        if isinstance(head, nn.Module) and not isinstance(head, nn.Identity):
            setattr(base, head_name, nn.Identity())
else:
    raise ValueError(MODEL_ID)

encoder = VisualEncoder(base, 320)

# Decoder Flickr30k được khởi tạo mới bằng SEED hiện tại; không nạp decoder
# MS COCO có vocabulary 10.080.
torch.manual_seed(SEED)
decoder = CaptionDecoder(len(vocab["tokens"]))
encoder.to(DEVICE)
decoder.to(DEVICE)

for module in encoder.modules():
    if isinstance(module, nn.modules.batchnorm._BatchNorm):
        module.eval()

enc_params = sum(p.numel() for p in encoder.parameters())
dec_params = sum(p.numel() for p in decoder.parameters())
assert enc_params == 5166356, (enc_params, 5166356)
assert dec_params == 5_084_960, dec_params
print("Images train/val       :", len(train_groups), "/", len(val_groups))
print("Nested image aliases   :", duplicate_alias_count, "(deduplicated)")
print("Encoder/decoder params :", enc_params, "/", dec_params)
print("Test                   : SEALED / NOT READ")

Images train/val       : 29000 / 1014
Nested image aliases   : 30014 (deduplicated)
Encoder/decoder params : 5166356 / 5084960
Test                   : SEALED / NOT READ


## Cell 3 — Teacher adaptation contract và paired views

In [3]:
TEACHER_ID = "T3"
TEACHER_NAME = "SigLIP ViT-B/16 adapted"
BASELINE_EXPERIMENT_ID = "F30K_S3_42"
BASELINE_RESULT_PATH = unique_named(BASELINE_EXPERIMENT_ID + "_RESULT.json")
TEACHER_RESULT_PATH = unique_named("F30K_T3_Adapt42_Full_RESULT.json")
DIRECT_CONTRACT_PATH = unique_named("DIRECT_LOGIT_CONTRACT.json")
baseline_result = json.loads(BASELINE_RESULT_PATH.read_text(encoding="utf-8"))
teacher_result = json.loads(TEACHER_RESULT_PATH.read_text(encoding="utf-8"))
direct_contract = json.loads(DIRECT_CONTRACT_PATH.read_text(encoding="utf-8"))

assert baseline_result["status"] == "complete"
assert baseline_result["model_id"] == "S3"
assert int(baseline_result["seed"]) == SEED == 42
assert baseline_result["test_set_status"] == "sealed"
assert baseline_result["test_set_used"] is False
assert teacher_result["status"] == "complete"
assert teacher_result["direct_logit_kd_ready"] is True
assert teacher_result["next_phase_authorized"] == EXPERIMENT_ID
assert teacher_result["test_set_status"] == "sealed"
assert teacher_result["test_set_used"] is False
assert direct_contract["status"] == "locked"
assert direct_contract["direct_logit_kd_allowed"] is True
assert direct_contract["teacher_logit_dimension"] == 7456
assert direct_contract["student_logit_dimension"] == 7456
assert direct_contract["vocabulary_sha256"] == sha256(VOCAB_PATH)

BASELINE_BEST_PATH = BASELINE_RESULT_PATH.parent / "best.pt"
TEACHER_BEST_PATH = TEACHER_RESULT_PATH.parent / "best.pt"
assert sha256(BASELINE_BEST_PATH) == baseline_result["artifacts"]["best.pt"]["sha256"]
assert sha256(TEACHER_BEST_PATH) == teacher_result["best_checkpoint_sha256"]

baseline_payload = torch.load(BASELINE_BEST_PATH, map_location="cpu", weights_only=False)
assert baseline_payload["experiment_id"] == BASELINE_EXPERIMENT_ID
encoder.load_state_dict(baseline_payload["encoder_state_dict"], strict=True)
decoder.load_state_dict(baseline_payload["decoder_state_dict"], strict=True)
del baseline_payload

from transformers import SiglipModel

teacher_source_candidates = []
for manifest_path in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        payload = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if payload.get("repo_id") == "google/siglip-base-patch16-224":
        teacher_source_candidates.append(manifest_path.parent)
assert len(teacher_source_candidates) == 1, teacher_source_candidates
TEACHER_MODEL_SOURCE = teacher_source_candidates[0]

TEACHER_SOURCE_CHECKPOINT = unique_named("teacher_c_seed42_best.pt")
assert sha256(TEACHER_SOURCE_CHECKPOINT) == teacher_result["teacher_checkpoint_sha256"]

class SiglipVisualAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(768, 256)
        self.position_embedding = nn.Parameter(torch.zeros(1, 196, 256))
        self.output_norm = nn.LayerNorm(256)
    def forward(self, patches):
        assert patches.shape[1:] == (196, 768), patches.shape
        return self.output_norm(
            self.projection(patches) + self.position_embedding
        )

teacher_full = SiglipModel.from_pretrained(
    str(TEACHER_MODEL_SOURCE), local_files_only=True
)
teacher_encoder = teacher_full.vision_model
del teacher_full
teacher_adapter = SiglipVisualAdapter()
teacher_decoder = CaptionDecoder(7456)

source_payload = torch.load(
    TEACHER_SOURCE_CHECKPOINT, map_location="cpu", weights_only=False
)
assert source_payload["schema_version"] == "phase3_teacher_c_seed42_checkpoint_v1"
teacher_encoder_state = teacher_encoder.state_dict()
teacher_encoder_state.update(source_payload["encoder_candidate_state"])
teacher_encoder.load_state_dict(teacher_encoder_state, strict=True)
teacher_adapter.load_state_dict(source_payload["adapter_state"], strict=True)
del source_payload

adapted_payload = torch.load(
    TEACHER_BEST_PATH, map_location="cpu", weights_only=False
)
assert adapted_payload["experiment_id"] == "F30K_T3_Adapt42_Full"
assert adapted_payload["vocabulary_sha256"] == sha256(VOCAB_PATH)
teacher_decoder.load_state_dict(
    adapted_payload["decoder_state_dict"], strict=True
)
del adapted_payload

teacher_encoder = teacher_encoder.to(DEVICE)
teacher_adapter = teacher_adapter.to(DEVICE)
teacher_decoder = teacher_decoder.to(DEVICE)
teacher_encoder.eval(); teacher_adapter.eval(); teacher_decoder.eval()
for module in [teacher_encoder, teacher_adapter, teacher_decoder]:
    for parameter in module.parameters():
        parameter.requires_grad_(False)

from torchvision.transforms import functional as TF

class DirectLogitKDDataset(Dataset):
    def __init__(self, rows):
        self.rows, self.epoch = rows, 0
    def set_epoch(self, epoch):
        self.epoch = int(epoch)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        image_id, captions = self.rows[index]
        caption = captions[(self.epoch + image_id % len(captions)) % len(captions)]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as source:
            image = source.convert("RGB")
            top, left, height, width = transforms.RandomResizedCrop.get_params(
                image, scale=(0.8, 1.0), ratio=(0.75, 1.3333333333333333)
            )
            image = TF.resized_crop(
                image, top, left, height, width, [224, 224],
                interpolation=InterpolationMode.BICUBIC,
            )
            if torch.rand(()) < 0.5:
                image = TF.hflip(image)
            tensor = TF.to_tensor(image)
        student_pixels = TF.normalize(tensor.clone(), MEAN, STD)
        teacher_pixels = TF.normalize(
            tensor, [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]
        )
        return (
            int(image_id), student_pixels, teacher_pixels,
            torch.tensor(caption, dtype=torch.long),
        )

def logit_collate(rows):
    image_ids, student_images, teacher_images, captions = zip(*rows)
    maximum = max(len(x) for x in captions)
    tokens = torch.zeros(len(captions), maximum, dtype=torch.long)
    mask = torch.zeros(len(captions), maximum, dtype=torch.bool)
    for index, values in enumerate(captions):
        tokens[index, :len(values)] = values
        mask[index, :len(values)] = True
    return (
        torch.tensor(image_ids), torch.stack(student_images),
        torch.stack(teacher_images), tokens, mask,
    )

train_set = DirectLogitKDDataset(train_groups)
def make_train_loader(epoch):
    train_set.set_epoch(epoch)
    return DataLoader(
        train_set, batch_size=8, shuffle=True,
        generator=torch.Generator().manual_seed(SEED + epoch),
        num_workers=2, pin_memory=True, persistent_workers=False,
        collate_fn=logit_collate,
    )

BASELINE_CIDER = float(baseline_result["best_validation_metrics"]["CIDEr_D"])
BASELINE_SHA256 = sha256(BASELINE_BEST_PATH)
TEACHER_ADAPTED_SHA256 = sha256(TEACHER_BEST_PATH)
TEMPERATURE = float(direct_contract["temperature"])
print("KD preflight           : PASSED")
print("Teacher / Student      :", TEACHER_NAME, "/", MODEL_NAME)
print("Vocabulary/logits      : 7456 / 7456")
print("Loss                   : 0.5 CE_human + 0.5 T^2 KL_logit, T=4")
print("Physical/accum/effect. : 8 / 8 / 64")
print("Baseline CIDEr-D       :", BASELINE_CIDER)
print("Test                   : SEALED / NOT READ")

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

KD preflight           : PASSED
Teacher / Student      : SigLIP ViT-B/16 adapted / TinyViT-5M
Vocabulary/logits      : 7456 / 7456
Loss                   : 0.5 CE_human + 0.5 T^2 KL_logit, T=4
Physical/accum/effect. : 8 / 8 / 64
Baseline CIDEr-D       : 0.5706915961103144
Test                   : SEALED / NOT READ


## Cell 4 — Beam-3 và metrics Validation

In [4]:
TOKENS = vocab["tokens"]
SPECIAL = {0, 1, 2, 3}

def ids_to_caption(ids):
    words = []
    for x in ids:
        x = int(x)
        if x == 2:
            break
        if x not in SPECIAL:
            words.append(TOKENS[x])
    text = " ".join(words)
    for p in [".", ",", "!", "?", ";", ":"]:
        text = text.replace(" " + p, p)
    return text.strip()

@torch.inference_mode()
def beam_decode(memory, beam_size=3, alpha=0.7):
    beams = [([1], 0.0, False)]
    for _ in range(30):
        active = [b for b in beams if not b[2]]
        if not active:
            break
        seq = torch.tensor([b[0] for b in active], device=DEVICE)
        mem = memory.expand(len(active), -1, -1)
        logits = decoder(seq, mem)[:, -1].float()
        logits[:, 0] = -torch.inf
        logits[:, 1] = -torch.inf
        logp = logits.log_softmax(-1)
        candidates = [b for b in beams if b[2]]
        for i, (tokens, score, _) in enumerate(active):
            values, indices = logp[i].topk(beam_size)
            for value, index in zip(values.tolist(), indices.tolist()):
                new = tokens + [index]
                candidates.append((new, score + value, index == 2))
        beams = sorted(candidates, key=lambda b: b[1] / (max(1, len(b[0]) - 1) ** alpha), reverse=True)[:beam_size]
    return max(beams, key=lambda b: b[1] / (max(1, len(b[0]) - 1) ** alpha))[0]

@torch.inference_mode()
def validate():
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    encoder.eval()
    decoder.eval()
    hypotheses = {}
    ce_sum, token_count = 0.0, 0
    for image_ids, images, tokens, mask in val_loader:
        images, tokens, mask = images.to(DEVICE), tokens.to(DEVICE), mask.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            memory = encoder(images)
            logits = decoder(tokens[:, :-1], memory, mask[:, :-1])
        targets = tokens[:, 1:]
        valid = mask[:, 1:]
        losses = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]),
                                 targets.reshape(-1), reduction="none").reshape_as(targets)
        ce_sum += float((losses * valid).sum())
        token_count += int(valid.sum())
        for j, image_id in enumerate(image_ids.tolist()):
            hypotheses[int(image_id)] = [ids_to_caption(beam_decode(memory[j:j+1]))]
    refs = {int(k): [{"caption": x} for x in v] for k, v in VAL_REFERENCES.items()}
    hyps = {int(k): [{"caption": v[0]}] for k, v in hypotheses.items()}
    tokenizer = PTBTokenizer()
    refs = tokenizer.tokenize(refs)
    hyps = tokenizer.tokenize(hyps)
    b, _ = Bleu(4).compute_score(refs, hyps)
    meteor = Meteor()
    try:
        m, _ = meteor.compute_score(refs, hyps)
    finally:
        close_method = getattr(meteor, "close", None)
        if callable(close_method):
            close_method()
    r, _ = Rouge().compute_score(refs, hyps)
    c, _ = Cider().compute_score(refs, hyps)
    return {"ce_loss": ce_sum / token_count, "BLEU_1": b[0], "BLEU_4": b[3],
            "METEOR": m, "ROUGE_L": r, "CIDEr_D": c}, hypotheses

## Cell 5 — Full Direct Logit-KD, Early Stopping và resume

In [5]:
MAX_EPOCHS = int(protocol["maximum_epochs"])
PATIENCE = int(protocol["early_stopping"]["patience"])
MIN_EPOCHS = int(protocol["early_stopping"]["minimum_completed_epochs"])
MIN_DELTA = float(protocol["minimum_cider_improvement"])
ACCUM = 8
ALPHA_CE = 0.5
ALPHA_KD = 0.5
steps_per_epoch = math.ceil(math.ceil(len(train_set) / 8) / ACCUM)
total_steps = MAX_EPOCHS * steps_per_epoch
warmup = max(1, round(0.05 * total_steps))

optimizer = torch.optim.AdamW([
    {"params": encoder.backbone.parameters(), "lr": 1e-4},
    {"params": list(encoder.projection.parameters()) +
               [encoder.visual_position_embedding] +
               list(encoder.output_norm.parameters()), "lr": 3e-4},
    {"params": decoder.parameters(), "lr": 3e-4},
], betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4)

def lr_lambda(step):
    if step < warmup:
        return max(1e-8, step / warmup)
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", init_scale=1024.0)
LAST = WORK_ROOT / "last.pt"
BEST = WORK_ROOT / "best.pt"
METRICS_CSV = WORK_ROOT / "metrics.csv"
CAPTIONS = WORK_ROOT / "best_validation_captions.json"

def state_cpu(module):
    return {key: value.detach().cpu() for key, value in module.state_dict().items()}

def save_checkpoint(path, epoch, best, stale, global_step, metrics):
    payload = {
        "schema_version": "f30k_direct_logit_kd_seed42_v1",
        "experiment_id": EXPERIMENT_ID, "model_id": MODEL_ID,
        "teacher_id": TEACHER_ID, "seed": SEED,
        "epoch": epoch, "best_cider": best, "stale_evaluations": stale,
        "global_step": global_step, "metrics": metrics,
        "encoder_state_dict": state_cpu(encoder),
        "decoder_state_dict": state_cpu(decoder),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all(),
        "baseline_checkpoint_sha256": BASELINE_SHA256,
        "teacher_adapted_checkpoint_sha256": TEACHER_ADAPTED_SHA256,
        "temperature": TEMPERATURE,
        "protocol_sha256": sha256(PROTOCOL_PATH),
        "data_contract_sha256": sha256(DATA_CONTRACT_PATH),
        "test_set_status": "sealed", "test_set_used": False,
    }
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)

start_epoch, best_cider, stale, global_step = 0, -math.inf, 0, 0
history = []
resume_candidates = sorted(INPUT_ROOT.rglob(f"{EXPERIMENT_ID}/last.pt"))
if resume_candidates:
    assert len(resume_candidates) == 1, resume_candidates
    payload = torch.load(resume_candidates[0], map_location="cpu", weights_only=False)
    assert payload["experiment_id"] == EXPERIMENT_ID
    assert payload["baseline_checkpoint_sha256"] == BASELINE_SHA256
    assert payload["teacher_adapted_checkpoint_sha256"] == TEACHER_ADAPTED_SHA256
    assert float(payload["temperature"]) == TEMPERATURE
    assert payload["test_set_used"] is False
    encoder.load_state_dict(payload["encoder_state_dict"], strict=True)
    decoder.load_state_dict(payload["decoder_state_dict"], strict=True)
    optimizer.load_state_dict(payload["optimizer_state_dict"])
    scheduler.load_state_dict(payload["scheduler_state_dict"])
    scaler.load_state_dict(payload["scaler_state_dict"])
    random.setstate(payload["python_rng_state"])
    np.random.set_state(payload["numpy_rng_state"])
    torch.set_rng_state(payload["torch_rng_state"])
    torch.cuda.set_rng_state_all(payload["cuda_rng_state_all"])
    start_epoch = int(payload["epoch"]) + 1
    best_cider = float(payload["best_cider"])
    stale = int(payload["stale_evaluations"])
    global_step = int(payload["global_step"])
    resume_root = resume_candidates[0].parent
    for name, destination in [
        ("metrics.csv", METRICS_CSV), ("best.pt", BEST),
        ("best_validation_captions.json", CAPTIONS),
    ]:
        source = resume_root / name
        if source.is_file():
            shutil.copy2(source, destination)
    if METRICS_CSV.exists():
        history = pd.read_csv(METRICS_CSV).to_dict("records")
    print("Exact resume           :", resume_candidates[0], "next epoch", start_epoch + 1)

run_started = time.time()
for epoch in range(start_epoch, MAX_EPOCHS):
    epoch_started = time.time()
    encoder.train(); decoder.train()
    teacher_encoder.eval(); teacher_adapter.eval(); teacher_decoder.eval()
    for module in encoder.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
    loader = make_train_loader(epoch)
    optimizer.zero_grad(set_to_none=True)
    ce_sum = kd_sum = 0.0
    token_count = 0
    for micro_step, (_, student_images, teacher_images, tokens, mask) in enumerate(loader):
        student_images = student_images.to(DEVICE)
        teacher_images = teacher_images.to(DEVICE)
        tokens, mask = tokens.to(DEVICE), mask.to(DEVICE)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            teacher_patches = teacher_encoder(
                pixel_values=teacher_images
            ).last_hidden_state
            teacher_memory = teacher_adapter(teacher_patches)
            teacher_logits = teacher_decoder(
                tokens[:, :-1], teacher_memory, mask[:, :-1]
            ).float()
        with torch.autocast("cuda", dtype=torch.float16):
            student_memory = encoder(student_images)
            student_logits = decoder(
                tokens[:, :-1], student_memory, mask[:, :-1]
            ).float()
            targets, valid = tokens[:, 1:], mask[:, 1:]
            ce_raw = F.cross_entropy(
                student_logits.reshape(-1, student_logits.shape[-1]),
                targets.reshape(-1), reduction="none",
            ).reshape_as(targets)
            ce = (ce_raw * valid).sum() / valid.sum()
            kd_raw = F.kl_div(
                F.log_softmax(student_logits / TEMPERATURE, dim=-1),
                F.softmax(teacher_logits / TEMPERATURE, dim=-1),
                reduction="none",
            ).sum(dim=-1)
            kd = (kd_raw * valid).sum() / valid.sum()
            loss = ALPHA_CE * ce + ALPHA_KD * (TEMPERATURE ** 2) * kd
        scaler.scale(loss / ACCUM).backward()
        ce_sum += float((ce_raw.detach() * valid).sum())
        kd_sum += float((kd_raw.detach() * valid).sum())
        token_count += int(valid.sum())
        final_micro = micro_step + 1 == len(loader)
        if (micro_step + 1) % ACCUM == 0 or final_micro:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(decoder.parameters()), 1.0
            )
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            global_step += 1
        if (micro_step + 1) % 200 == 0 or final_micro:
            print(
                f"epoch={epoch+1:02d} batch={micro_step+1:04d}/{len(loader):04d} "
                f"CE={ce_sum/token_count:.6f} KL={kd_sum/token_count:.6f}",
                flush=True,
            )
    print(f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | VALIDATION START", flush=True)
    metrics, hypotheses = validate()
    metrics.update({
        "epoch": epoch + 1,
        "train_ce": ce_sum / token_count,
        "train_kl": kd_sum / token_count,
        "train_weighted_loss": (
            ALPHA_CE * ce_sum / token_count +
            ALPHA_KD * (TEMPERATURE ** 2) * kd_sum / token_count
        ),
        "global_step": global_step,
        "elapsed_hours": (time.time() - run_started) / 3600,
    })
    history.append(metrics)
    pd.DataFrame(history).to_csv(METRICS_CSV, index=False)
    improved = metrics["CIDEr_D"] > best_cider + MIN_DELTA
    if improved:
        best_cider, stale = metrics["CIDEr_D"], 0
    else:
        stale += 1
    save_checkpoint(LAST, epoch, best_cider, stale, global_step, metrics)
    if improved:
        save_checkpoint(BEST, epoch, best_cider, stale, global_step, metrics)
        CAPTIONS.write_text(
            json.dumps(
                {"references": VAL_REFERENCES, "hypotheses": hypotheses},
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )
    print(
        f"Epoch {epoch+1:02d} | CIDEr-D={metrics['CIDEr_D']:.6f} | "
        f"delta_baseline={metrics['CIDEr_D']-BASELINE_CIDER:+.6f} | "
        f"stale={stale}/{PATIENCE} | seconds={time.time()-epoch_started:.1f}",
        flush=True,
    )
    if epoch + 1 >= MIN_EPOCHS and stale >= PATIENCE:
        print("Early stopping         : TRIGGERED")
        break

assert BEST.exists() and LAST.exists()
best_payload = torch.load(BEST, map_location="cpu", weights_only=False)
best_metrics = best_payload["metrics"]
summary = {
    "schema_version": "f30k_direct_logit_kd_result_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete", "experiment_id": EXPERIMENT_ID,
    "teacher_id": TEACHER_ID, "teacher_name": TEACHER_NAME,
    "student_id": MODEL_ID, "student_name": MODEL_NAME, "seed": SEED,
    "method": "Direct Logit-KD", "temperature": TEMPERATURE,
    "loss": "0.5 CE_human + 0.5 T^2 KL_teacher_student",
    "best_epoch": int(best_payload["epoch"]) + 1,
    "best_validation_metrics": best_metrics,
    "baseline_validation_cider_d": BASELINE_CIDER,
    "delta_validation_cider_d": float(best_metrics["CIDEr_D"]) - BASELINE_CIDER,
    "selection_metric": "Validation CIDEr-D",
    "maximum_epochs": MAX_EPOCHS, "early_stopping_enabled": True,
    "next_phase_authorized": "F30K_T3S3_LogitKD_123_2026",
    "test_set_status": "sealed", "test_set_used": False,
    "artifacts": {},
}
for path in [BEST, LAST, METRICS_CSV, CAPTIONS]:
    summary["artifacts"][path.name] = {
        "bytes": path.stat().st_size, "sha256": sha256(path)
    }
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
result_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
manifest = {
    "schema_version": "f30k_kd_manifest_v1",
    "experiment_id": EXPERIMENT_ID, "status": "complete",
    "artifacts": [
        {
            "relative_path": path.name, "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in [result_path, BEST, LAST, METRICS_CSV, CAPTIONS]
    ],
    "test_set_status": "sealed", "test_set_used": False,
}
(WORK_ROOT / "MANIFEST.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Status                 : COMPLETE")
print("Best Validation        :", best_metrics)
print("Delta vs baseline      :", summary["delta_validation_cider_d"])
print("Next                   :", summary["next_phase_authorized"])
print("Held-out Test          : SEALED / NOT USED")

epoch=01 batch=0200/3625 CE=2.504145 KL=0.036416
epoch=01 batch=0400/3625 CE=2.495086 KL=0.035953
epoch=01 batch=0600/3625 CE=2.493486 KL=0.035464
epoch=01 batch=0800/3625 CE=2.486454 KL=0.035016
epoch=01 batch=1000/3625 CE=2.486130 KL=0.034591
epoch=01 batch=1200/3625 CE=2.487233 KL=0.034198
epoch=01 batch=1400/3625 CE=2.492820 KL=0.033818
epoch=01 batch=1600/3625 CE=2.496563 KL=0.033438
epoch=01 batch=1800/3625 CE=2.496068 KL=0.033084
epoch=01 batch=2000/3625 CE=2.497262 KL=0.032770
epoch=01 batch=2200/3625 CE=2.498506 KL=0.032467
epoch=01 batch=2400/3625 CE=2.500423 KL=0.032194
epoch=01 batch=2600/3625 CE=2.502381 KL=0.031941
epoch=01 batch=2800/3625 CE=2.504450 KL=0.031698
epoch=01 batch=3000/3625 CE=2.505024 KL=0.031476
epoch=01 batch=3200/3625 CE=2.505443 KL=0.031262
epoch=01 batch=3400/3625 CE=2.504578 KL=0.031057
epoch=01 batch=3600/3625 CE=2.504877 KL=0.030872
epoch=01 batch=3625/3625 CE=2.504504 KL=0.030844
Epoch 01/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 354384.53 tokens per second.
PTBTokenizer tokenized 12957 tokens at 118688.50 tokens per second.


{'testlen': 10902, 'reflen': 10750, 'guess': [10902, 9888, 8874, 7860], 'correct': [7459, 3703, 1756, 838]}
ratio: 1.0141395348836266
Epoch 01 | CIDEr-D=0.541390 | delta_baseline=-0.029302 | stale=0/4 | seconds=1122.6
epoch=02 batch=0200/3625 CE=2.533983 KL=0.027170
epoch=02 batch=0400/3625 CE=2.550662 KL=0.027157
epoch=02 batch=0600/3625 CE=2.548228 KL=0.027020
epoch=02 batch=0800/3625 CE=2.550336 KL=0.026967
epoch=02 batch=1000/3625 CE=2.559837 KL=0.026850
epoch=02 batch=1200/3625 CE=2.554547 KL=0.026749
epoch=02 batch=1400/3625 CE=2.550035 KL=0.026647
epoch=02 batch=1600/3625 CE=2.554513 KL=0.026554
epoch=02 batch=1800/3625 CE=2.560237 KL=0.026489
epoch=02 batch=2000/3625 CE=2.555057 KL=0.026402
epoch=02 batch=2200/3625 CE=2.552331 KL=0.026308
epoch=02 batch=2400/3625 CE=2.552705 KL=0.026233
epoch=02 batch=2600/3625 CE=2.552635 KL=0.026150
epoch=02 batch=2800/3625 CE=2.551787 KL=0.026079
epoch=02 batch=3000/3625 CE=2.553358 KL=0.025996
epoch=02 batch=3200/3625 CE=2.556368 KL=0.02593

PTBTokenizer tokenized 74030 tokens at 344064.94 tokens per second.
PTBTokenizer tokenized 14118 tokens at 139476.13 tokens per second.


{'testlen': 12051, 'reflen': 11595, 'guess': [12051, 11037, 10023, 9009], 'correct': [7915, 3922, 1798, 830]}
ratio: 1.0393272962482933
Epoch 02 | CIDEr-D=0.542418 | delta_baseline=-0.028274 | stale=0/4 | seconds=435.1
epoch=03 batch=0200/3625 CE=2.567673 KL=0.024512
epoch=03 batch=0400/3625 CE=2.547183 KL=0.024451
epoch=03 batch=0600/3625 CE=2.535612 KL=0.024370
epoch=03 batch=0800/3625 CE=2.532154 KL=0.024327
epoch=03 batch=1000/3625 CE=2.524296 KL=0.024235
epoch=03 batch=1200/3625 CE=2.528894 KL=0.024172
epoch=03 batch=1400/3625 CE=2.528820 KL=0.024120
epoch=03 batch=1600/3625 CE=2.523598 KL=0.024070
epoch=03 batch=1800/3625 CE=2.526067 KL=0.024031
epoch=03 batch=2000/3625 CE=2.529068 KL=0.023986
epoch=03 batch=2200/3625 CE=2.529416 KL=0.023957
epoch=03 batch=2400/3625 CE=2.529702 KL=0.023919
epoch=03 batch=2600/3625 CE=2.530786 KL=0.023872
epoch=03 batch=2800/3625 CE=2.528940 KL=0.023826
epoch=03 batch=3000/3625 CE=2.529048 KL=0.023788
epoch=03 batch=3200/3625 CE=2.529249 KL=0.0237

PTBTokenizer tokenized 74030 tokens at 339902.45 tokens per second.
PTBTokenizer tokenized 13555 tokens at 133459.08 tokens per second.


... (19 outputs left out)

PTBTokenizer tokenized 74030 tokens at 364289.39 tokens per second.
PTBTokenizer tokenized 13741 tokens at 121615.13 tokens per second.


{'testlen': 11667, 'reflen': 11387, 'guess': [11667, 10653, 9639, 8625], 'correct': [8063, 4052, 1919, 903]}
ratio: 1.0245894441028345
Epoch 13 | CIDEr-D=0.597105 | delta_baseline=+0.026413 | stale=3/4 | seconds=434.9
epoch=14 batch=0200/3625 CE=2.373988 KL=0.018255
epoch=14 batch=0400/3625 CE=2.358790 KL=0.018303
epoch=14 batch=0600/3625 CE=2.355470 KL=0.018364
epoch=14 batch=0800/3625 CE=2.353773 KL=0.018356
epoch=14 batch=1000/3625 CE=2.350162 KL=0.018320
epoch=14 batch=1200/3625 CE=2.342343 KL=0.018291
epoch=14 batch=1400/3625 CE=2.337831 KL=0.018273
epoch=14 batch=1600/3625 CE=2.331152 KL=0.018257
epoch=14 batch=1800/3625 CE=2.335017 KL=0.018268
epoch=14 batch=2000/3625 CE=2.334071 KL=0.018267
epoch=14 batch=2200/3625 CE=2.330311 KL=0.018254
epoch=14 batch=2400/3625 CE=2.332571 KL=0.018250
epoch=14 batch=2600/3625 CE=2.331800 KL=0.018235
epoch=14 batch=2800/3625 CE=2.332283 KL=0.018227
epoch=14 batch=3000/3625 CE=2.331379 KL=0.018226
epoch=14 batch=3200/3625 CE=2.333705 KL=0.01823

PTBTokenizer tokenized 74030 tokens at 342653.78 tokens per second.
PTBTokenizer tokenized 13169 tokens at 115530.42 tokens per second.


{'testlen': 11103, 'reflen': 11011, 'guess': [11103, 10089, 9075, 8061], 'correct': [7755, 3938, 1883, 872]}
ratio: 1.008355281082462
Epoch 14 | CIDEr-D=0.616857 | delta_baseline=+0.046166 | stale=4/4 | seconds=436.6
Early stopping         : TRIGGERED
Status                 : COMPLETE
Best Validation        : {'ce_loss': 2.7545091490472857, 'BLEU_1': 0.6952422907488374, 'BLEU_4': 0.279519800720964, 'METEOR': 0.22151968028521943, 'ROUGE_L': np.float64(0.49543244311706297), 'CIDEr_D': np.float64(0.617055344048548), 'epoch': 10, 'train_ce': 2.405513508156057, 'train_kl': 0.0192257359569103, 'train_weighted_loss': 1.3565626417333108, 'global_step': 4540, 'elapsed_hours': 1.3976881170272828}
Delta vs baseline      : 0.04636374793823361
Next                   : F30K_T3S3_LogitKD_123_2026
Held-out Test          : SEALED / NOT USED
